In [ ]:
import numpy as np
from manim import *

import icm_anim as anim

In [ ]:
class PhaseAccumulation(Scene):
    """The chapter's frequency ramp turned into phase two ways. The correct
    oscillator adds up the area under f(t) one sliver at a time. The naive
    one takes f(t) times the whole elapsed time: a single rectangle that
    pretends the oscillator always ran at its current frequency. Below, the
    frequency each one actually produces."""

    def construct(self):
        T_END = 4.0
        N_RECT = 40
        DT = T_END / N_RECT

        def f(t):
            # 440 Hz held, ramped up to 880 Hz, then held
            return float(np.interp(t, [0, 1, 3, 4], [440, 440, 880, 880]))

        def heard_naive(t):
            # the slope of f(t) * t is f(t) + f'(t) * t
            ramp = 220.0 if 1.0 <= t < 3.0 else 0.0
            return f(t) + ramp * t

        axis_cfg = {"color": anim.IRON, "include_ticks": False,
                    "stroke_width": 1.5}
        top = Axes(x_range=[0, T_END, 1], y_range=[0, 1200, 500],
                   x_length=10.5, y_length=2.4, axis_config=axis_cfg,
                   tips=False).move_to([0.45, 1.65, 0])
        bot = Axes(x_range=[0, T_END, 1], y_range=[0, 1700, 500],
                   x_length=10.5, y_length=2.4, axis_config=axis_cfg,
                   tips=False).move_to([0.45, -2.2, 0])

        top_title = Text("frequency control f(t)", font_size=24)
        top_title.next_to(top, UP, buff=0.18).align_to(top, LEFT)
        bot_title = Text("frequency you hear", font_size=24)
        bot_title.next_to(bot, UP, buff=0.18).align_to(bot, LEFT)
        time_label = Text("time", font_size=22)
        time_label.next_to(bot.x_axis.get_end(), DOWN, buff=0.2)

        def ytick(ax, v):
            lab = MathTex(str(v)).scale(0.62)
            lab.next_to(ax.c2p(0, v), LEFT, buff=0.14)
            tick = Line(ax.c2p(0, v), ax.c2p(0, v) + RIGHT * 0.1,
                        color=anim.IRON, stroke_width=1.5)
            return VGroup(lab, tick)

        ticks = VGroup(ytick(top, 440), ytick(top, 880),
                       ytick(bot, 440), ytick(bot, 880), ytick(bot, 1540))

        control = top.plot(f, x_range=[0, T_END, 0.01], color=anim.IRON,
                           stroke_width=3, use_smoothing=False)

        # pass 1: the correct oscillator, slivers of area under the curve
        c1 = ValueTracker(0.0)
        fill1 = ValueTracker(0.55)

        def slivers():
            c = c1.get_value()
            g = VGroup()
            for i in range(N_RECT):
                x0 = i * DT
                if x0 >= c - 1e-9:
                    break
                x1 = min(x0 + DT, c)
                h = f(x0)
                g.add(Polygon(top.c2p(x0, 0), top.c2p(x1, 0), top.c2p(x1, h),
                              top.c2p(x0, h), stroke_color=anim.GOLD,
                              stroke_width=1, fill_color=anim.GOLD,
                              fill_opacity=fill1.get_value()))
            return g

        area = always_redraw(slivers)
        heard_ok = always_redraw(lambda: bot.plot(
            f, x_range=[0, max(c1.get_value(), 1e-3), 0.01],
            color=anim.RED, stroke_width=3.2, use_smoothing=False))
        pen1 = always_redraw(lambda: Dot(
            top.c2p(c1.get_value(), f(c1.get_value())),
            color=anim.GOLD, radius=0.07))
        ok_label = MathTex(r"\text{phase} = \text{area under } f").scale(0.7)
        ok_label.move_to(top.c2p(0.08, 1080), aligned_edge=LEFT)
        ok_legend = Text("correct", font_size=24, color=anim.RED)
        ok_legend.next_to(bot.c2p(2.0, 660), DOWN + RIGHT, buff=0.12)

        # pass 2: the naive oscillator, one rectangle as tall as f is now
        c2 = ValueTracker(0.0)

        def naive_rect():
            c = max(c2.get_value(), 1e-3)
            h = f(c)
            return Polygon(top.c2p(0, 0), top.c2p(c, 0), top.c2p(c, h),
                           top.c2p(0, h), stroke_color=anim.BLUE,
                           stroke_width=2.5, fill_color=anim.BLUE,
                           fill_opacity=0.16)

        rect = always_redraw(naive_rect)
        heard_bad = always_redraw(lambda: bot.plot(
            heard_naive, x_range=[0, max(c2.get_value(), 1e-3), 0.005],
            discontinuities=[1.0, 3.0], dt=0.002, color=anim.BLUE,
            stroke_width=3.2, use_smoothing=False))
        def jumps():
            c = c2.get_value()
            g = VGroup()
            for tj, lo, hi in ((1.0, 440, 660), (3.0, 880, 1540)):
                if c >= tj:
                    g.add(DashedLine(bot.c2p(tj, lo), bot.c2p(tj, hi),
                                     color=anim.BLUE, stroke_width=2,
                                     dash_length=0.08))
            return g

        jump_lines = always_redraw(jumps)
        pen2 = always_redraw(lambda: Dot(
            top.c2p(c2.get_value(), f(c2.get_value())),
            color=anim.BLUE, radius=0.07))
        bad_label = MathTex(r"\text{phase} = f(t) \cdot t",
                            color=anim.BLUE).scale(0.7)
        bad_label.move_to(top.c2p(0.08, 1080), aligned_edge=LEFT)
        bad_legend = Text("naive", font_size=24, color=anim.BLUE)
        bad_legend.next_to(bot.c2p(2.0, 1100), UP + LEFT, buff=0.12)

        def sweep(tracker, seconds):
            self.play(tracker.animate(rate_func=linear).set_value(T_END),
                      run_time=seconds)

        self.play(FadeIn(top), FadeIn(bot), FadeIn(top_title),
                  FadeIn(bot_title), FadeIn(time_label), FadeIn(ticks),
                  run_time=1.0)
        self.play(Create(control), run_time=1.0)

        self.add(area, heard_ok, pen1)
        self.play(FadeIn(ok_label), FadeIn(ok_legend), run_time=0.5)
        sweep(c1, 5.5)
        self.wait(0.6)

        self.play(fill1.animate.set_value(0.18), FadeOut(pen1),
                  FadeOut(ok_label), run_time=0.6)
        self.add(rect, heard_bad, jump_lines, pen2)
        self.bring_to_front(heard_ok)
        self.play(FadeIn(bad_label), FadeIn(bad_legend), run_time=0.5)
        sweep(c2, 5.5)
        self.wait(1.5)

anim.show(PhaseAccumulation)